In [ ]:
# ml4fmri cross-validation benchmark on the MDD DIRECT ICA-53 timecourses
%matplotlib inline
import numpy as np
from ml4fmri import cvbench

In [ ]:
# load the packed super_clean archive and keep only 'full' subjects (Diagnosis+Sex+Age present)
d = np.load("data/mdd_ica53_tc_superclean.npz")
full = d["full"]
X = d["ica53_tc"][full]                 # (samples, time, 53)  -> cvbench DATA
y = d["Diagnosis"][full].astype(int)    # 0 = HC, 1 = MDD       -> cvbench LABELS

# rows are ordered in site/diagnosis blocks; cvbench's test loader is unshuffled, so without
# this the per-batch test AUC hits single-class batches and returns nan. Shuffle to break blocks.
rng = np.random.default_rng(42)
perm = rng.permutation(len(y))
X, y = X[perm], y[perm]
print("X:", X.shape, "| y:", y.shape, "| label counts [HC, MDD]:", np.bincount(y))

In [ ]:
# 10-fold CV benchmark. models: 'meanMLP' | 'lite' | 'ts' | 'all' | ['name', ...]
report = cvbench(X, y, models="all", n_folds=10)

In [ ]:
# test AUC boxplots per model across folds
report.plot_scores()
report.save()